<a href="https://colab.research.google.com/github/OlhaZahrebelna/certflow-rag-assistant/blob/main/src/ingestion/chunker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pyyaml

In [2]:
import re
from pathlib import Path
import yaml


def load_markdown_document(file_path: str) -> tuple[dict, str]:
    """
    Load metadata and content from a Markdown document.

    Expected format:

    ---
    document_id: ACD-KB-001
    title: Account Data Certification Overview
    ...
    ---

    # Account Data Certification Overview
    ...
    """

    path = Path(file_path)

    text = path.read_text(encoding="utf-8")

    # Split YAML front matter from Markdown content
    parts = text.split("---", 2)

    if len(parts) != 3:
        raise ValueError(
            f"File {file_path} does not contain valid YAML front matter."
        )

    metadata_text = parts[1]
    content = parts[2].strip()

    metadata = yaml.safe_load(metadata_text)

    return metadata, content

In [3]:
def split_into_sections(content: str) -> list[dict]:
    """
    Split Markdown document by level-2 headings (##).
    """

    pattern = r"(?m)^##\s+(.+)$"

    matches = list(re.finditer(pattern, content))

    sections = []

    for i, match in enumerate(matches):

        section_title = match.group(1).strip()

        start = match.end()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(content)

        section_content = content[start:end].strip()

        sections.append(
            {
                "section_title": section_title,
                "content": section_content,
            }
        )

    return sections

In [4]:
def create_chunks(
    metadata: dict,
    sections: list[dict]
) -> list[dict]:

    chunks = []

    for index, section in enumerate(sections):

        chunk_id = (
            f"{metadata['document_id']}-"
            f"chunk-{index + 1:03d}"
        )

        chunk = {
            "chunk_id": chunk_id,

            "content": section["content"],

            "metadata": {
                **metadata,

                "section": section["section_title"],

                "section_number": index + 1
            }
        }

        chunks.append(chunk)

    return chunks

In [5]:
def process_document(file_path: str) -> list[dict]:

    metadata, content = load_markdown_document(file_path)

    sections = split_into_sections(content)

    chunks = create_chunks(
        metadata=metadata,
        sections=sections
    )

    return chunks

In [ ]:
from src.ingestion.chunker import process_document

chunks = process_document(
    "data/raw/source_markdown/01_account_certification_overview.md"
)

for chunk in chunks:
    print("=" * 80)
    print(chunk["chunk_id"])
    print(chunk["metadata"]["section"])
    print(chunk["content"])